# 07 - Advanced Analysis: Normalization, Scaling, And Fits

This notebook introduces analysis workflows that transform or fit a series of CVs. It starts by making copied CV lists for physical dimensionless normalization, standardized current, and `i/ip0` normalized current, then moves into scan-rate fitting and templates for Sevcik, peak-potential, trumpet, Nicholson, plateau-current, FOWA, and Tafel-style analysis.

The examples are workflow demonstrations. The code is intentionally compact, but the assumptions are not automatic: replace placeholder physical constants, fit windows, diffusion coefficients, and mechanistic assumptions before using the outputs for interpretation. After these sections, the main CV-series advanced helpers are represented here; `fit_rate()` is included as the bridge from FOWA result tables into concentration-order and rate-scaling fits.


## Import eCAT And Set Paths

The setup is intentionally short now that notebooks 00-02 have introduced the pattern.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name

import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
plt.rcParams["figure.max_open_warning"] = 0
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))


eCAT version: 0.1.0b4
Example data: examples/data/fe_phoh_cv
Text files: 13


## Load, Inspect, And Select Analysis Series

The advanced-analysis examples need reference-corrected CVs, scan-rate metadata, and electrode area metadata. The reference correction uses Fc/Fc+ keyword matching. One trace in this sample set has a less reliable self-reference at high PhOH concentration, so the import includes a small `reference map`: before filtering, object `12` is the 2.8 M PhOH CV and object `11` is the 1 M PhOH CV, so `12: 11` tells eCAT to use the 1 M reference assignment for the 2.8 M trace.

`electrode diameter` is included during import so eCAT can store electrode area on each CV; `normalize()` can then use that metadata automatically. After import, the multiscan file is removed with a public `segments` filter so the analysis series contain only ordinary 3-segment CVs.


In [2]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "reference mode": "keyword",
    "reference keyword": "Fc",
    "reference guess": 0.4,
    "reference label": "Fc/Fc+",
    "reference map": {12: 11},
    "electrode diameter": 0.3,
    "print": True,
})
cvs = e.filter(cvs, {"segments": 3}, {"print": False})


Searching recursively through:
 examples/data/fe_phoh_cv
13 .txt files found.



Reference correction:
  Mode: keyword
  Keyword: Fc
  Guess: 0.4 V
  Folder reference: MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs.txt = 0.4665 V
  Usage:
    folder/ancestor reference: 2
    self-referenced successfully: 10
    explicit reference map: 1

[Conditions] Exp Type: CV, Solvent: MeCN, Compounds: 0.1 M TBAPF₆
[Reference] Label: Fc/Fc+


,Gas,Compounds,Scan Window,Scan Rate,Segments,Reference Shift,Reference Mode,Reference Source
[0],Ar,,"[-1.2, 1]",100 mV/s,3,0.467,folder,[1]
[1],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.2, 1]",100 mV/s,3,0.467,folder,[1]
[2],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",100 mV/s,3,0.467,self,[2]
[3],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",25 mV/s,3,0.467,self,[3]
[4],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",500 mV/s,3,0.466,self,[4]
[5],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",1 V/s,3,0.466,self,[5]
[6],Ar,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.7, 1]",50 mV/s,3,0.467,self,[6]
[7],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me","[-1.2, 1]",100 mV/s,3,0.467,self,[7]
[8],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me, 100 mM PhOH","[-1.2, 1]",100 mV/s,3,0.466,self,[8]
[9],CO2,"3 mM Fc, 1 mM Fe-tpyPY2Me, 560 mM PhOH","[-1.2, 1]",100 mV/s,3,0.468,self,[9]


In [3]:
blank = e.filter(cvs, {
    "gas": "Ar",
    "compounds": ["Fc", "Fe-tpyPY2Me"],
}, {"logic": "AND", "mode": "exclude", "print": False})[0]

fe_100 = e.filter(cvs, {
    "scan rate": 0.1,
    "scan window": [-1.2, 1],
    "compounds": "Fe-tpyPY2Me",
}, {"logic": "AND", "print": False})

fe_ar = e.filter(cvs, {
    "gas": "Ar",
    "compounds": "Fe-tpyPY2Me",
}, {"logic": "AND", "print": False})

co2_cvs = e.filter(cvs, {
    "gas": "CO2",
}, {"print": False})

co2_only = e.filter(co2_cvs, {
    "compounds": "PhOH",
}, {"mode": "exclude", "print": False})[0]

phoh_co2 = e.filter(co2_cvs, {
    "compounds": "PhOH",
}, {"print": False})
high_phoh = [obj for obj in phoh_co2 if "2.8MPhOH" in obj.name][0]
co2_titration = [co2_only] + phoh_co2

scan_series = e.filter(fe_ar, {
    "scan window": [-1.7, 1],
}, {"print": False})
scan_series = e.sort(scan_series, "scan rate", {"print": False})

target = phoh_co2[0]

e.show_objects(scan_series);


[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Scan Rate
[0],25 mV/s
[1],50 mV/s
[2],100 mV/s
[3],500 mV/s
[4],1 V/s


## Plot The Scan-Rate Series With A Colorbar

Before fitting a scan-rate series, plot it. This is the quickest check that the same feature is being tracked across the series, the scan windows match, and no trace has an obvious import or current-direction problem. Here `multiplot()` uses a colorbar instead of a long legend because scan rate is an ordered variable.


In [4]:
ax = e.multiplot(scan_series, {
    "print": False,
    "title": "Ar scan-rate series",
    "subtitle": "Gradient color by scan-rate metadata",
    "legend mode": "colorbar",
    "colorbar tick labels": "all",
    "gradient scale": "index",
    "colorbar height": 2.5,
})
# ax.figure.savefig(EXPORT_DIR / "advanced_scan_rate_series.png", dpi=300, bbox_inches="tight")

## Physical Dimensionless Normalization With `normalize()`

`normalize()` returns copied CVs with dimensionless potential and/or current axes. For a scan-rate series, eCAT can read the scan rate from each CV and use the electrode area stored during import. The values below are placeholders: replace `D`, `C`, `n`, `E0`, and `temperature` with values that are appropriate for your system.

When `print` is on for a series, eCAT prints the symbolic equation once, reports the shared mode outside the table, and shows one parameter table with group-index columns.

In [5]:
normalized_scan_series = e.normalize(scan_series, {
    "mode": "homogeneous",
    "D": 1e-5,
    "C": "1 mM",
    "n": 2,
    "E0": 0,
    "temperature": 298,
    "print": True,
})
e.multiplot(normalized_scan_series, {
    "title": "Dimensionless scan-rate series",
});

CV normalization summary:
Mode: homogeneous


<IPython.core.display.Math object>

<IPython.core.display.Math object>

,0,1,2,3,4
Parameter,,,,,
n,2,2,2,2,2
T / K,298,298,298,298,298
E0 / V,0,0,0,0,0
D / cm2 s-1,1e-05,1e-05,1e-05,1e-05,1e-05
C* / mol cm-3,1e-06,1e-06,1e-06,1e-06,1e-06
C* input,1 mM,1 mM,1 mM,1 mM,1 mM
S / cm2,0.07069,0.07069,0.07069,0.07069,0.07069
ν / V s-1,0.025,0.05,0.1,0.5,1


## Standardize Current With `scale_current()`

`scale_current()` returns copied CVs whose raw current columns have been multiplied by scale factors. This is useful when you want to compare waveform shape after matching a reference peak or manually chosen current scale. It is not the same as `i/ip0`: the current is still in current units, just scaled.

Here the CO2/no-PhOH trace is used as the reference CV for the CO2 titration. eCAT extracts the reference current with peak-current settings, computes scale factors, and returns scaled copies that can be passed straight to `multiplot()`.

In [6]:
standardized_titration = e.scale_current(fe_100, {
    "reference cv": fe_100[0],
    "reference mode": "single",
    "segment": 2,
    "guess potential": 0,
    "print": True,
})

e.multiplot(standardized_titration, {
    "title": "CO2 titration with standardized current",
    "subtitle": "scale_current() keeps current units",
});

Current scaling summary:
[Conditions] Exp Type: CV, Solvent: MeCN, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.2, 1], Scan Rate: 100 mV/s, Segments: 3, IR Comp Percent: 100 %


,Gas,Compounds,Scale Factor
[0],Ar,,1.000000
[1],CO2,,0.816700
[2],CO2,100 mM PhOH,0.685300
[3],CO2,560 mM PhOH,0.726800
[4],CO2,1 M PhOH,0.818900
[5],CO2,2.8 M PhOH,1.134000


## Normalize Current As `i/ip0` With `normalize_current()`

`normalize_current()` returns copied CVs with a stored `i/ip0` axis. This is the dimensionless current-ratio workflow used when a non-catalytic reference current is the natural scale. Compared with `scale_current()`, the y-axis is no longer current in amperes; it is a ratio to `ip0`.

For this PhOH titration, use the CO2/no-PhOH CV as the non-catalytic reference. The normalized copies still behave like CVs, so `multiplot()` automatically uses the normalized y-axis unless you explicitly ask for raw `Current`.

In [7]:
normalized_titration = e.normalize_current(co2_titration, {
    "reference cv": co2_only,
    "guess potential": -1.5,
    "print": True,
})

e.multiplot(normalized_titration, {
    "title": "CO2 titration normalized to i/ip0",
    "legend": True,
});

Current Normalization:
CVs: 5
ip0: -5.117e-05 A
Source: reference CV: MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs


## Fit Peak Current With `fit_peak_current`

`fit_peak_current()` extracts a peak current from each CV, resolves the varying x-axis automatically, and fits the resulting trend. For scan-rate series, the default x transform is useful for Randles-Sevcik-style behavior because peak current commonly scales with the square root of scan rate. Use this as a fast diagnostic before deciding whether a full Sevcik analysis is justified.


In [8]:
fit = e.fit_peak_current(scan_series, {
    "plot all": True,
    "plot fit": True,
    "fit label": True,
    "segment": 1,
    "print": True,
    "return stats": True,
})

[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Plot Label,Scan Rate
[0],25 mV/s,25 mV/s
[1],50 mV/s,50 mV/s
[2],100 mV/s,100 mV/s
[3],500 mV/s,500 mV/s
[4],1 V/s,1 V/s


Fit Model:


,Field,Value
0,Model,linear
1,Equation,y = m x + b
2,Residual,direct
3,X Range,0.158114 to 1
4,Fit Points,5
5,R²,0.999273
6,RMSE,0.988654
7,m,-113.878 ± 1.77315
8,b,-4.13243 ± 1.02628


## Fit Peak Potential With `fit_peak_potential`

`fit_peak_potential()` tracks where a selected feature appears as a series changes. In a scan-rate series, shifts in peak potential can flag kinetic limitations, uncompensated resistance, adsorption, or simply a feature-picking problem. The function is intentionally parallel to `fit_peak_current()`: it collects one value per CV, plots the trend, and returns a fit result object that can be displayed with `e.show()`.

Here the two main segments are fitted separately. `guess potential` gives the peak picker a starting feature; in real use, adjust it after inspecting the overlay above.


In [9]:
ep_fit = e.fit_peak_potential(scan_series, {
    "plot all": True,
    "plot": True,
    "plot fit": True,
    "fit label": True,
    "print": True,
    "segments": [1, 2],
    "guess potential": -1.5,
    "follow E1/2": True,
    "return stats": True,
})


[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Plot Label,Scan Rate
[0],25 mV/s,25 mV/s
[1],50 mV/s,50 mV/s
[2],100 mV/s,100 mV/s
[3],500 mV/s,500 mV/s
[4],1 V/s,1 V/s


Fit Model:


,Field,Seg 1 Ep,Seg 2 Ep,Seg 1-2 E1/2
0,Model,linear,linear,linear
1,Equation,y = m x + b,y = m x + b,y = m x + b
2,Residual,direct,direct,direct
3,X Range,-1.60206 to 0,-1.60206 to 0,-1.60206 to 0
4,Fit Points,5,5,5
5,R²,0.748847,0.155664,0.934232
6,RMSE,0.00170391,0.00102322,0.00044861
7,m,-0.00488344 ± 0.00163282,-0.000729215 ± 0.000980519,-0.00280633 ± 0.000429891
8,b,-1.48791 ± 0.00168898,-1.36421 ± 0.00101425,-1.42606 ± 0.000444678


## Trumpet Analysis With `trumpet_analysis`

Trumpet analysis follows paired anodic and cathodic peak potentials across scan rate. As scan rate increases, quasireversible electron-transfer systems often show widening peak separation; fitting those two branches can provide electron-transfer descriptors such as transfer-coefficient estimates and, when a diffusion coefficient is supplied, an apparent heterogeneous rate constant.

Use this when you have a clean redox couple, reliable paired peaks, and enough scan rates to make the two branches meaningful. The `segment` pair is supplied as `[1, 2]`, which tells eCAT to use adjacent forward/reverse segments from the same redox feature.


In [10]:
trumpet = e.trumpet_analysis(scan_series, {
    "plot all": True,
    "plot fit": True,
    "fit label": True,
    "print": True,
    "segments": [1, 2],
    "guess potential": -1.5,
    "D": 1e-5,
})

e.show(trumpet);


[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Plot Label,Scan Rate
[0],25 mV/s,25 mV/s
[1],50 mV/s,50 mV/s
[2],100 mV/s,100 mV/s
[3],500 mV/s,500 mV/s
[4],1 V/s,1 V/s


Trumpet Analysis Summary:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

,Parameter,Value
0,α,6.193
1,β,-35.24
2,k0,5.990e-17 cm/s
3,Warning,α/β may not be reliable: reverse branch slope does not have the expected positive sign; alpha is outside the physical 0-1 range; beta is outside the physical 0-1 range; branch intersection falls outside the fitted log(scan rate) window.


Fit Model:


,Field,Seg 1,Seg 2
0,Model,linear,linear
1,Equation,y = m x + b,y = m x + b
2,Residual,direct,direct
3,X Range,-1.602 to 0,-1.602 to 0
4,Fit Points,5,5
5,R²,0.8142,0.3991
6,RMSE,0.001374,0.0006201
7,m,-0.004774 ± 0.001317,-0.0008388 ± 0.0005943
8,b,-1.488 ± 0.001362,-1.364 ± 0.0006147


Fit Model:


,Field,Seg 1,Seg 2
0,Model,linear,linear
1,Equation,y = m x + b,y = m x + b
2,Residual,direct,direct
3,X Range,-1.602 to 0,-1.602 to 0
4,Fit Points,5,5
5,R²,0.8142,0.3991
6,RMSE,0.001374,0.0006201
7,m,-0.004774 ± 0.001317,-0.0008388 ± 0.0005943
8,b,-1.488 ± 0.001362,-1.364 ± 0.0006147


## Nicholson Analysis With `nicholson_analysis`

Nicholson analysis estimates heterogeneous electron-transfer kinetics from peak separation. It is most appropriate for reversible-to-quasireversible redox couples where peak separation is dominated by electron-transfer kinetics rather than bad referencing, uncompensated resistance, adsorption, or background subtraction. The diffusion coefficient is required because the fitted kinetic scale depends on it.

Nicholson uses a single starting segment and pairs it with the next segment, so this example uses `"segment": 1` to analyze segments 1 and 2 together. The `D` value is a placeholder; replace it before treating the reported `k0` as physical. This teaching cell suppresses the automated iR/background warning because that assumption is stated here explicitly.


In [11]:
nicholson = e.nicholson_analysis(scan_series, {
    "plot all": True,
    "plot fit": True,
    "print": True,
    "segment": 1,
    "guess potential": -1.5,
    "D": 2e-5,
    "warn ir drop": False,
    'exclude invalid delta ep': False,
})


Nicholson Analysis Equation:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Nicholson Parameters:


,Parameter,Value
0,Fit Model,origin
1,D,2.000e-05 cm^2/s
2,n,1
3,T,298 kK
4,ψ Source,agarwal table
5,Valid nΔEp Range,61 to 212 mV



Nicholson Analysis Summary:


,Setting,Value
0,Included Points,5 / 5
1,Excluded Points,0
2,k0,0.00403 cm/s
3,Intercept,0
4,R²,-71.08
5,k0 Point Mean,0.00819 cm/s
6,k0 Point Median,0.005913 cm/s
7,k0 Point Std,0.005638 cm/s



Nicholson Analysis Data:


,name,scan rate / V/s,temperature / K,Ep1 / V,Ep2 / V,E1/2 / V,ΔEp / mV,nΔEp / mV,ψ,Nicholson x / s/cm,k0 point / cm/s,included,exclusion reason,psi source
0,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_to_1V_25mVs,0.025,298,-1.482,-1.362,-1.422,120,120,0.356,127.9,0.002784,Yes,,agarwal table
1,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_to_1V_50mVs,0.05,298,-1.481,-1.364,-1.422,117,117,0.378,90.41,0.004181,Yes,,agarwal table
2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_to_1V_100mVs,0.1,298,-1.481,-1.364,-1.422,117,117,0.378,63.93,0.005913,Yes,,agarwal table
3,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_to_1V_500mVs,0.5,298,-1.486,-1.364,-1.425,122,122,0.343,28.59,0.012,Yes,,agarwal table
4,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_to_1V_1Vs,1,298,-1.489,-1.364,-1.426,125,125,0.325,20.22,0.01608,Yes,,agarwal table


## Sevcik Analysis With `sevcik_analysis`

`sevcik_analysis()` is the more explicit scan-rate workflow. It connects peak current to scan rate through the Sevcik relationship, so the result is only meaningful when the series is diffusion-controlled and the concentration, electrode area, electron count, and temperature assumptions are appropriate. This example is a template: the code runs, but the physical constants should be replaced with values for the system being interpreted.


In [12]:
sevcik = e.sevcik_analysis(scan_series, {
    "plot all": True,
    "plot fit": True,
    "fit label": True,
    "num electrons": 1,
    "segments":[1,2],
    "guess potential": -1.5,
    "legend": True
})

[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Plot Label,Scan Rate
[0],25 mV/s,25 mV/s
[1],50 mV/s,50 mV/s
[2],100 mV/s,100 mV/s
[3],500 mV/s,500 mV/s
[4],1 V/s,1 V/s


Sevcik Analysis Summary:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

,Parameter,Value
0,n,1
1,T,298 K
2,S,0.07069 cm^2
3,C*,1e-06 mol/cm^3
4,Scan dependence,0.5


Sevcik Fit Results:


,series,Fit,R2,RMSE,Fit Points,fit x min,fit x max,Diffusion Coefficient
0,Seg 1,y = -113.9x -4.129,0.999277,0.985925,5,0.158114,1.0,3.590e-05 cm^2/s
1,Seg 2,y = 104.3x +4.974,0.999320,0.875479,5,0.158114,1.0,3.010e-05 cm^2/s


## Plateau Current With `plateau_current()`

`plateau_current()` estimates a catalytic limiting or plateau current and can convert it into an observed rate constant when enough supporting assumptions are available. The cleanest use is a catalytic scan-rate series: eCAT can check whether the apparent plateau current is scan-rate independent before using it. Here we only have one 2.8 M PhOH CV, so this is a single-trace demonstration rather than a validated plateau-current workflow.

The 2.8 M trace is compared against the CO2/no-PhOH CV as the non-catalytic reference. `validate plateau` is turned off because one catalytic CV cannot test scan-rate independence; for publication-quality use, provide a scan-rate series or a manual `ilim` after inspecting the wave.


In [13]:
plateau = e.plateau_current(high_phoh, {
    "non-catalytic cv": co2_only,
    "guess potential": -1.5,
    "non-catalytic guess potential": -1.5,
    "validate plateau": False,
    "plot all": True,
    "print": True,
})

plateau


### Plateau Current Summary ###
formula mode: normalized
valid plateau: True
ilim: 0.0001502 A (peak_current)
ip0: -5.117e-05 A


,cv / cvs used,ilim,ilim source,ilim average method,valid plateau,plateau warning,catalytic scan rates,catalytic sqrt scan rates,plateau subset cvs,plateau subset scan rates,plateau slope,plateau intercept,plateau slope metric,ip0,ip0 Source,ip0 scan rate,ip0 sqrt scan rate slope,ip0 fit r2,formula mode,formula,D,C,electrode area,catalyst electrons,turnover electrons,kobs
0,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_2.8MPhOH_-1.2_to_1V_100mVs,0.000150,peak_current,mean,True,,[0.1],[0.31622776601683794],['MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_2.8MPhOH_-1.2_to_1V_100mVs'],[0.1],0.000000,0.000150,0.000000,-0.000051,non-catalytic cv,0.100000,None,nan,normalized,kobs = (0.446 |ilim/ip0| sqrt(nFv_ip0/RT))^2 / n',None,None,0.070686,1.000000,1.000000,6.674052


{'data':                                        cv / cvs used     ilim   ilim source  \
 0  MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_2.8MP...  0.00015  peak_current   
 
   ilim average method  valid plateau plateau warning catalytic scan rates  \
 0                mean           True                                [0.1]   
 
   catalytic sqrt scan rates  \
 0     [0.31622776601683794]   
 
                                   plateau subset cvs  \
 0  [MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_2.8M...   
 
   plateau subset scan rates  ...  ip0 sqrt scan rate slope  ip0 fit r2  \
 0                     [0.1]  ...                      None         NaN   
 
    formula mode                                            formula     D  \
 0    normalized  kobs = (0.446 |ilim/ip0| sqrt(nFv_ip0/RT))^2 / n'  None   
 
       C electrode area  catalyst electrons turnover electrons      kobs  
 0  None       0.070686                 1.0                1.0  6.674052  
 
 [1 rows x 26 columns],
 'summ

## FOWA With `fowa()`

Foot-of-the-wave analysis focuses on the low-overpotential region of a catalytic wave. It is useful when the catalytic response has a region where kinetic information can be extracted before mass transport, substrate depletion, or wave-shape complications dominate. The important user choices are the non-catalytic reference, the redox potential, and the fit region; the printed summary is meant to make those assumptions easy to audit.


In [14]:
fowa_table = e.fowa(phoh_co2, {
    "plot all": True,
    "non-catalytic cv": fe_ar[0],
    "redox mode": "manual",
    "redox potential": -1.47,
    "fit basis": "y",
    "fit range": [0.1, 0.5],
    "diagnostic y axis": "i/ip0",
    "min fit points": 50,
    "min r2": .95,
})

[Conditions] Exp Type: CV, Solvent: MeCN, Gas: CO₂, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.2, 1], Scan Rate: 100 mV/s, Segments: 3, IR Comp Percent: 100 %

### FOWA Summary ###


,Field,Value
0,Reference CV,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs
1,ip0 Source,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs
2,Redox Source,manual (-1.47 V)
3,Redox Potential,-1.47
4,Segment,1
5,Segment Selection,default: segment 1
6,Background Correction,tangent
7,Fit Range,"[0.1, 0.5]"
8,Mechanism,EC'
9,ip0,-4.056e-05


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

,Compounds,Background Tangent,Wave Range,Fit Points,FOWA Fit,R2,Status,kobs
0,100 mM PhOH,y = 2.6071e-06x - 2.78394e-06,"[-1.213, -1.521]",67,y = 45.3349x + 0.0853366,0.989,ok,1.594e+03
1,560 mM PhOH,y = 2.36291e-06x - 3.28446e-06,"[-1.21, -1.536]",69,y = 69.1028x + 0.0806315,0.9943,ok,3.704e+03
2,1 M PhOH,y = 3.0778e-06x - 2.1115e-06,"[-1.2145, -1.5745]",62,y = 68.7767x + 0.0673961,0.9971,ok,3.669e+03
3,2.8 M PhOH,y = -3.44142e-07x - 4.26493e-06,"[-1.224, -1.71]",56,y = 18.0337x + 0.0524495,0.9964,ok,2.523e+02


## Fit FOWA Rates With `fit_rate()`

The FOWA table is still a normal result table, so it can be passed directly into `fit_rate()`. Here the x-axis is auto-resolved from the PhOH concentration metadata and the y-axis is the FOWA-derived `kobs`. A log-log fit is a quick way to estimate the apparent concentration order across the titration.

Use this as an exploratory fit: check the FOWA status/R2 values, decide whether all concentrations belong in the same regime, and adjust `fit indices` before interpreting the slope.


In [15]:
fowa_rate_fit = e.fit_rate(fowa_table, {
    "metric": "kobs",
    "species": "PhOH",
    "transform mode": "log-log",
    "plot": True,
    "plot fit": True,
    "fit label": True,
    "print": True,
})

fowa_rate_fit.table


Fit Model:


,Field,Value
0,Model,linear
1,Equation,y = m x + b
2,Residual,direct
3,X Range,-1 to 0.4472
4,Fit Points,4
5,R²,0.2168
6,RMSE,0.4209
7,m,-0.422 ± 0.5672
8,b,3.1 ± 0.3188


,Compounds,Reference CV,ip0 Source,Redox Source,Reference Ep,Redox Mode,Redox Delta E,Redox Potential,Catalytic Ecat/2,Ecat/2 - E1/2,...,y label,y raw,y adjusted,y0,y mode,x transformed,y transformed,x transform,y transform,y transform note
0,100 mM PhOH,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,manual,None,manual,None,-1.47,None,None,...,kobs,1594.154885,1594.154885,1594.154885,raw,-1.000000,3.202531,log10,log10,
1,560 mM PhOH,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,manual,None,manual,None,-1.47,None,None,...,kobs,3703.876211,3703.876211,1594.154885,raw,-0.251812,3.568656,log10,log10,
2,1 M PhOH,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,manual,None,manual,None,-1.47,None,None,...,kobs,3668.996649,3668.996649,1594.154885,raw,0.000000,3.564547,log10,log10,
3,2.8 M PhOH,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,manual,None,manual,None,-1.47,None,None,...,kobs,252.252249,252.252249,1594.154885,raw,0.447158,2.401835,log10,log10,


## Manual Tafel-Style Analysis

`tafel_analysis()` converts a `TOFmax` value, a thermodynamic potential, and a catalyst redox potential into a Tafel-style `log10(TOF)` vs overpotential curve. A single CV gives one curve; a list of CVs gives one curve per input and reuses the same labels, colors, gradients, legends, and colorbars as `multiplot()`.

The redox potential below is the Fe redox potential used in the FOWA section. Set `CO2_TO_CO_THERMO_POTENTIAL` from the literature value and reference convention you want to use before interpreting the overpotential axis.


In [16]:
FE_REDOX_POTENTIAL = -1.47
CO2_TO_CO_THERMO_POTENTIAL = -1.20  # V vs Fc/Fc+; replace with your chosen literature convention

manual_tafel = e.tafel_analysis(
    phoh_co2[0],
    TOF_max=1000,
    thermodynamic_potential=CO2_TO_CO_THERMO_POTENTIAL,
    redox_potential=FE_REDOX_POTENTIAL,
    options={"color": "black"},
)
plt.gca().set_title("Manual Tafel template")

manual_tafel["summary"]


,Index,Label,TOFmax,Temperature,Thermodynamic Potential,Redox Potential
0,0,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_100mM...,1000.0,298,-1.2,-1.47


## Tafel-Style Curves From All FOWA Rates

The FOWA table gives one `TOFmax` value for each PhOH concentration. Passing the whole `phoh_co2` list and the matching `TOFmax` values lets `tafel_analysis()` draw the full concentration series in one call.

Because this uses the same backend style helpers as `multiplot()`, the familiar overlay options work here too: labels, label alterations, discrete colors, gradients, colorbars, legend placement, titles, subtitles, and scale bars.


In [17]:
fowa_rates = fowa_table.table.attrs["full_results_df"].reset_index(drop=True)

fowa_tafel = e.tafel_analysis(
    phoh_co2,
    TOF_max=fowa_rates["TOFmax"],
    thermodynamic_potential=CO2_TO_CO_THERMO_POTENTIAL,
    redox_potential=FE_REDOX_POTENTIAL,
    options={
        "title": "Tafel-style curves from FOWA rates",
        "subtitle": "PhOH concentration series",
        "gradient by": "concentration",
        "gradient species": "PhOH",
        "legend mode": "colorbar",
        "colorbar tick labels": "endpoints",
    },
)

fowa_tafel["summary"]


,Index,Label,TOFmax,Temperature,Thermodynamic Potential,Redox Potential
0,0,100 mM PhOH,1594.154885,298,-1.2,-1.47
1,1,560 mM PhOH,3703.876211,298,-1.2,-1.47
2,2,1 M PhOH,3668.996649,298,-1.2,-1.47
3,3,2.8 M PhOH,252.252249,298,-1.2,-1.47
